
# CNN Transfer Baseline — DeepFakeDetector

**Goal:** Fine-tune a pretrained CNN to classify images as **Real** vs **AI-generated** (deepfakes).  
This notebook is a **beginner-friendly, end-to-end baseline** that you can run locally or in Colab.

**Backbones supported:** `EfficientNet-B0` (default) and `ResNet-50` (toggle below).  
**Metrics:** Accuracy, F1, AUROC.  
**Extras:** Optional saliency map for interpretability, model checkpointing, and clear instructions.

> ✅ This file is the deliverable requested by the issue: `notebooks/cnn_transfer_baseline.ipynb`.



## 0. Setup

- You need Python 3.10+ recommended.
- If running the notebook locally, install dependencies with the cell below.
- If running in **Google Colab**, the cell will also work; GPU is recommended.

> Dataset (expected folder structure):
```
data/
  train/
    real/
    fake/
  val/
    real/
    fake/
```
- Each folder contains images (JPG/PNG). You may rename `fake` to `ai` if you like—just update the `CLASS_NAMES` below.


In [ ]:

# If you are on Colab or a fresh environment, uncomment and run:
# %pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
# %pip install scikit-learn matplotlib tqdm pillow captum


In [ ]:

import os, time, copy, math
from pathlib import Path

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models

from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
import matplotlib.pyplot as plt
import numpy as np

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", DEVICE)

# ---- Beginner-friendly switches ----
DATA_DIR = Path("data")   # change if your dataset lives elsewhere
USE_MODEL = "efficientnet_b0"  # options: "efficientnet_b0", "resnet50"
FREEZE_BACKBONE = True         # freeze feature extractor first
UNFREEZE_EPOCH = 3             # unfreeze from this epoch (set None to keep frozen)
NUM_EPOCHS = 5                 # increase for real training
BATCH_SIZE = 32
LR = 1e-3
WEIGHT_DECAY = 1e-4
NUM_WORKERS = 2

# Class name mapping (folder names under train/ and val/). Adjust if your folder uses "ai" instead of "fake".
CLASS_NAMES = ["real", "fake"]

# Where to save best checkpoints & logs
ARTIFACTS_DIR = Path("artifacts")
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)



## 1. Data Loading and Augmentations

- We use `ImageFolder` so the folder names define classes.
- Basic augmentations help generalization.


In [ ]:

IM_SIZE = 224  # EfficientNet-B0 / ResNet-50 default

train_tfms = transforms.Compose([
    transforms.Resize((IM_SIZE, IM_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1, hue=0.02),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

val_tfms = transforms.Compose([
    transforms.Resize((IM_SIZE, IM_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

train_dir = DATA_DIR / "train"
val_dir = DATA_DIR / "val"
assert train_dir.exists(), f"Missing folder: {train_dir}"
assert val_dir.exists(), f"Missing folder: {val_dir}"

train_ds = datasets.ImageFolder(train_dir, transform=train_tfms)
val_ds = datasets.ImageFolder(val_dir, transform=val_tfms)

# Make sure classes match expected names (warn if not)
print("Detected classes:", train_ds.classes)
assert train_ds.classes == CLASS_NAMES, f"Expected classes {CLASS_NAMES}, got {train_ds.classes}"

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

print(f"Train images: {len(train_ds)}, Val images: {len(val_ds)}")



## 2. Build the Model (Transfer Learning)

Pick a pretrained backbone and replace the classification head with a 2‑class head.


In [ ]:

def build_model(name: str, num_classes: int = 2, freeze_backbone: bool = True):
    if name == "efficientnet_b0":
        from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights
        weights = EfficientNet_B0_Weights.IMAGENET1K_V1
        model = efficientnet_b0(weights=weights)
        in_features = model.classifier[1].in_features
        model.classifier[1] = nn.Linear(in_features, num_classes)
        feature_modules = [model.features]
    elif name == "resnet50":
        from torchvision.models import resnet50, ResNet50_Weights
        weights = ResNet50_Weights.IMAGENET1K_V2
        model = resnet50(weights=weights)
        in_features = model.fc.in_features
        model.fc = nn.Linear(in_features, num_classes)
        feature_modules = [model.conv1, model.layer1, model.layer2, model.layer3, model.layer4]
    else:
        raise ValueError("Unknown model: " + name)

    if freeze_backbone:
        for m in feature_modules:
            for p in m.parameters():
                p.requires_grad = False

    return model

model = build_model(USE_MODEL, num_classes=2, freeze_backbone=FREEZE_BACKBONE).to(DEVICE)
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()),
                        lr=LR, weight_decay=WEIGHT_DECAY)
print(model.__class__.__name__, "built. Trainable params:",
      sum(p.numel() for p in model.parameters() if p.requires_grad))



## 3. Training & Validation Loop

- Trains with the head (and optionally unfreezes the backbone later).
- Tracks Accuracy, F1, and AUROC.
- Saves the **best** model checkpoint by AUROC.


In [ ]:

def run_epoch(model, loader, optimizer=None):
    is_train = optimizer is not None
    model.train(is_train)
    all_logits, all_y = [], []
    running_loss = 0.0

    for imgs, labels in loader:
        imgs = imgs.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)

        with torch.set_grad_enabled(is_train):
            logits = model(imgs)
            loss = criterion(logits, labels)

            if is_train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

        running_loss += loss.item() * imgs.size(0)
        all_logits.append(logits.detach().cpu())
        all_y.append(labels.detach().cpu())

    all_logits = torch.cat(all_logits)
    all_y = torch.cat(all_y).numpy()
    probs = torch.softmax(all_logits, dim=1)[:, 1].numpy()  # prob(fake)
    preds = (probs >= 0.5).astype(int)

    epoch_loss = running_loss / len(loader.dataset)
    acc = accuracy_score(all_y, preds)
    f1 = f1_score(all_y, preds, zero_division=0)
    try:
        auroc = roc_auc_score(all_y, probs)
    except ValueError:
        auroc = float("nan")
    return epoch_loss, acc, f1, auroc

best_auroc = -1.0
best_path = ARTIFACTS_DIR / f"best_{USE_MODEL}.pt"

history = {"train": [], "val": []}

for epoch in range(1, NUM_EPOCHS+1):
    # Unfreeze backbone if requested
    if UNFREEZE_EPOCH is not None and epoch == UNFREEZE_EPOCH:
        print(f"Unfreezing backbone at epoch {epoch}...")
        for p in model.parameters():
            p.requires_grad = True
        optimizer = optim.AdamW(model.parameters(), lr=LR/10, weight_decay=WEIGHT_DECAY)  # smaller LR for all params

    t0 = time.time()
    tr_loss, tr_acc, tr_f1, tr_auroc = run_epoch(model, train_loader, optimizer)
    va_loss, va_acc, va_f1, va_auroc = run_epoch(model, val_loader, optimizer=None)
    dt = time.time() - t0

    history["train"].append((tr_loss, tr_acc, tr_f1, tr_auroc))
    history["val"].append((va_loss, va_acc, va_f1, va_auroc))

    print(f"Epoch {epoch:02d} | {dt:.1f}s "
          f"| train loss {tr_loss:.4f} acc {tr_acc:.3f} f1 {tr_f1:.3f} auroc {tr_auroc:.3f} "
          f"| val loss {va_loss:.4f} acc {va_acc:.3f} f1 {va_f1:.3f} auroc {va_auroc:.3f}")

    # Save best
    if not math.isnan(va_auroc) and va_auroc > best_auroc:
        best_auroc = va_auroc
        torch.save({"model_state": model.state_dict(),
                    "model_name": USE_MODEL,
                    "class_names": CLASS_NAMES}, best_path)
        print(f"✅ Saved new best checkpoint to {best_path} (val AUROC={best_auroc:.3f})")



## 4. Curves


In [ ]:

# Plot Accuracy & AUROC (validation)
val_acc = [x[1] for x in history["val"]]
val_auroc = [x[3] for x in history["val"]]

plt.figure()
plt.plot(range(1, len(val_acc)+1), val_acc, marker='o')
plt.title("Validation Accuracy")
plt.xlabel("Epoch"); plt.ylabel("Accuracy"); plt.grid(True)
plt.show()

plt.figure()
plt.plot(range(1, len(val_auroc)+1), val_auroc, marker='o')
plt.title("Validation AUROC")
plt.xlabel("Epoch"); plt.ylabel("AUROC"); plt.grid(True)
plt.show()



## 5. Optional: Simple Saliency Map (Captum)

This helps **explain** which pixels most influence the prediction.  
Run after training. Provide a path to a single image.


In [ ]:

# Optional saliency using Captum (install in the first cell if missing)
try:
    from captum.attr import Saliency
    HAVE_CAPTUM = True
except Exception as e:
    HAVE_CAPTUM = False
    print("Captum not installed. Install with: %pip install captum")

def show_saliency(img_path):
    assert HAVE_CAPTUM, "Install captum first."
    model.eval()
    from PIL import Image
    import torchvision.transforms.functional as TF

    pil = Image.open(img_path).convert("RGB")
    x = val_tfms(pil).unsqueeze(0).to(DEVICE).requires_grad_(True)
    logits = model(x)
    target_class = logits.argmax(dim=1).item()
    sal = Saliency(model)
    attributions = sal.attribute(x, target=target_class)  # shape [1,3,H,W]
    attr = attributions.abs().detach().squeeze().mean(0).cpu().numpy()

    fig, ax = plt.subplots(1,3, figsize=(12,4))
    ax[0].imshow(pil); ax[0].set_title("Image"); ax[0].axis("off")
    ax[1].imshow(attr, cmap="inferno"); ax[1].set_title("Saliency"); ax[1].axis("off")
    ax[2].imshow(pil); ax[2].imshow(attr, cmap="inferno", alpha=0.5); ax[2].set_title("Overlay"); ax[2].axis("off")
    plt.show()

# Example usage:
# show_saliency('data/val/fake/example.png')



## 6. Inference Helper


In [ ]:

IDX_TO_NAME = {i: n for i, n in enumerate(CLASS_NAMES)}

def load_best(model_path=best_path):
    ckpt = torch.load(model_path, map_location=DEVICE)
    model = build_model(ckpt["model_name"], num_classes=len(ckpt["class_names"]), freeze_backbone=False)
    model.load_state_dict(ckpt["model_state"])
    model.to(DEVICE).eval()
    return model, ckpt["class_names"]

@torch.inference_mode()
def predict_image(img_path, model=None):
    if model is None:
        model, _ = load_best(best_path)
    from PIL import Image
    pil = Image.open(img_path).convert("RGB")
    x = val_tfms(pil).unsqueeze(0).to(DEVICE)
    logits = model(x)
    prob_fake = torch.softmax(logits, dim=1)[0,1].item()
    pred = int(prob_fake >= 0.5)
    return {"prob_fake": prob_fake, "pred": pred, "label": IDX_TO_NAME[pred]}

# Example:
# predict_image('data/val/real/example.jpg')



## 7. Record Results (Fill After Training)

- **Backbone:** EfficientNet-B0 or ResNet-50  
- **Freeze > Unfreeze epoch:** e.g., 1–2 epochs frozen, unfreeze at 3  
- **Val Accuracy:** …  
- **Val F1:** …  
- **Val AUROC:** …  
- **Observations:** … (e.g., overfitting signs, which backbone worked better, any class imbalance tweaks)

> Tip: If your dataset is imbalanced, consider using `WeightedRandomSampler` or class weights in `CrossEntropyLoss`.



## 8. (Optional) Write a `requirements.txt`

Run this cell once to generate a minimal `requirements.txt` alongside artifacts.


In [ ]:

req = '''torch
torchvision
torchaudio
scikit-learn
matplotlib
tqdm
pillow
captum
'''
with open('requirements.txt', 'w') as f:
    f.write(req.strip() + "\n")
print("Wrote requirements.txt")
